# PesoLens img256 — export only (no training)

Exports the verified Kaggle v1 `best.pt` (imgsz 256, 100 epochs, sha256 `cfa7b65d…`) to a
full-integer INT8 TFLite with **Ultralytics 8.4.76**, checks it against the firmware contract,
and measures the INT8 model's own mAP on the validation split. Nothing after setup may raise.

In [ ]:
# Preflight
import socket
from pathlib import Path
socket.create_connection(('pypi.org', 443), timeout=10).close(); print('Internet: OK')
assert len(list(Path('/kaggle/input').rglob('data.yaml'))) == 2, 'pesolens-july7 not attached'
BEST_SRC = next(Path('/kaggle/input').rglob('best.pt'), None)
assert BEST_SRC, 'pesolens-img256-v1-best not attached'
import hashlib
sha = hashlib.sha256(BEST_SRC.read_bytes()).hexdigest(); print(BEST_SRC, sha)
assert sha == 'cfa7b65d49661e6fc1968917c7034e5363ad0d9e34eb50913371d47711ff5efe', 'wrong best.pt'


In [ ]:
!pip install -q ultralytics==8.4.76 pillow-heif "onnx2tf>=1.26.3,<1.29.0" "tf_keras<=2.19.0" "sng4onnx>=1.0.1" "onnx_graphsurgeon>=0.3.26" "ai-edge-litert>=1.2.0" onnxruntime "onnxslim>=0.1.82"
import importlib
for m in ['onnx2tf', 'onnx_graphsurgeon', 'sng4onnx', 'tf_keras', 'tensorflow', 'ai_edge_litert', 'onnxruntime', 'onnxslim']:
    mod = importlib.import_module(m); print(m, getattr(mod, '__version__', '?'))
import ultralytics; print('ultralytics', ultralytics.__version__)
assert ultralytics.__version__ == '8.4.76'

In [ ]:
from pathlib import Path
from collections import Counter
import shutil, zipfile
from PIL import Image
from pillow_heif import register_heif_opener
register_heif_opener()

# Firmware class order — DO NOT reorder.
TARGET = ['20', '50', '100', '500', '1000']
target_index = {d: i for i, d in enumerate(TARGET)}

# Roboflow 9-class index -> denomination string
NINE_TO_DENOM = {
    0: '100', 1: '1000', 2: '50', 3: '500',
    4: '100', 5: '1000', 6: '20', 7: '50', 8: '500',
}
REMAP = {i: target_index[NINE_TO_DENOM[i]] for i in range(9)}
print('9->5 label remap:', REMAP)

OUT = Path('/kaggle/working/pesolens_merged')
IMG_SIZE = 256  # was 320 in the approved model; the ONLY change

In [ ]:
# Kaggle auto-extracts uploaded zips, so locate each dataset by its data.yaml
yamls = sorted(Path('/kaggle/input').rglob('data.yaml'))
print('Found dataset roots:')
for y in yamls:
    print(' ', y.parent)
roots = [y.parent for y in yamls]
assert len(roots) == 2, f'Expected 2 dataset roots, found {len(roots)}: {roots}'

In [ ]:
if OUT.exists():
    shutil.rmtree(OUT)
for split in ['train', 'valid', 'test']:
    (OUT / split / 'images').mkdir(parents=True, exist_ok=True)
    (OUT / split / 'labels').mkdir(parents=True, exist_ok=True)

IMG_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.heic', '.heif'}

def remap_label(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        parts[0] = str(REMAP[int(float(parts[0]))])
        out.append(' '.join(parts))
    return ('\n'.join(out) + '\n') if out else ''

for zi, root in enumerate(roots):
    prefix = f'z{zi}'
    for split in ['train', 'valid', 'test']:
        idir = root / split / 'images'
        ldir = root / split / 'labels'
        if not idir.exists():
            continue
        for img in idir.iterdir():
            if img.suffix.lower() not in IMG_EXT:
                continue
            stem = img.stem
            if img.suffix.lower() in {'.heic', '.heif'}:
                dst_img = OUT / split / 'images' / f'{prefix}_{stem}.jpg'
                Image.open(img).convert('RGB').save(dst_img, quality=92)
            else:
                dst_img = OUT / split / 'images' / f'{prefix}_{stem}{img.suffix.lower()}'
                shutil.copy2(img, dst_img)
            lbl = ldir / f'{stem}.txt'
            dst_lbl = OUT / split / 'labels' / f'{prefix}_{stem}.txt'
            dst_lbl.write_text(remap_label(lbl.read_text()) if lbl.exists() else '')

print('Merge done.')

In [ ]:
EXPECTED_IMAGES = {'train': 3891, 'valid': 567, 'test': 566}
EXPECTED_HIST = Counter({0: 644, 1: 1675, 2: 1582, 3: 1377, 4: 1374})

hist = Counter()
for split in ['train', 'valid', 'test']:
    imgs = list((OUT / split / 'images').glob('*'))
    lbls = list((OUT / split / 'labels').glob('*.txt'))
    print(f'{split:5s}: {len(imgs)} imgs, {len(lbls)} labels')
    assert len(imgs) == EXPECTED_IMAGES[split], (split, len(imgs))
    for lp in lbls:
        for line in lp.read_text().splitlines():
            if line.strip():
                hist[int(line.split()[0])] += 1

print('Class histogram:', {TARGET[k]: hist[k] for k in sorted(hist)})
assert hist == EXPECTED_HIST, (hist, EXPECTED_HIST)
print('AUDIT PASSED — safe to train.')

In [ ]:
import yaml
data = {
    'path': str(OUT),
    'train': 'train/images',
    'val': 'valid/images',
    'test': 'test/images',
    'nc': len(TARGET),
    'names': TARGET,
}
(OUT / 'data.yaml').write_text(yaml.safe_dump(data, sort_keys=False))
print((OUT / 'data.yaml').read_text())

In [ ]:
# From here on NOTHING may raise: if the notebook errors, Kaggle discards every output file.
import traceback, json
REPORT = {}
def safe(name, fn):
    try:
        REPORT[name] = {'ok': True, 'result': fn()}
    except Exception as e:
        traceback.print_exc()
        REPORT[name] = {'ok': False, 'error': f'{type(e).__name__}: {e}'}
    print(f'[{name}]', 'OK' if REPORT[name]['ok'] else 'FAILED -> ' + REPORT[name]['error'])

In [ ]:
import shutil
from ultralytics import YOLO
EXP_DIR = Path('/kaggle/working/export_ul8476')
def _export():
    EXP_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy2(BEST_SRC, EXP_DIR / 'best.pt')
    p = YOLO(str(EXP_DIR / 'best.pt')).export(format='tflite', int8=True, imgsz=IMG_SIZE,
                                              data=str(OUT / 'data.yaml'))
    print('EXPORTED', p)
    return [str(q) for q in EXP_DIR.rglob('*.tflite')]
safe('export_ul8476', _export)


In [ ]:
# Firmware contract (firmware/src/inference.cpp): input int8 [1,256,256,3] scale 1/255 zp -128,
# output int8 [1,9,1344], every op registered in the MicroMutableOpResolver.
import numpy as np, collections
FW_OPS = {'ADD','CONCATENATION','CONV_2D','DEPTHWISE_CONV_2D','DEQUANTIZE','FULLY_CONNECTED',
          'HARD_SWISH','LEAKY_RELU','LOGISTIC','MAX_POOL_2D','MEAN','MUL','PAD','QUANTIZE',
          'RESHAPE','RESIZE_BILINEAR','RESIZE_NEAREST_NEIGHBOR','SHAPE','SLICE','SOFTMAX',
          'SPLIT','SPLIT_V','SQUEEZE','STRIDED_SLICE','SUB','TRANSPOSE','UNPACK'}
A = (IMG_SIZE//8)**2 + (IMG_SIZE//16)**2 + (IMG_SIZE//32)**2
def _check_one(path):
    try:
        from ai_edge_litert.interpreter import Interpreter
    except ImportError:
        import tensorflow as tf; Interpreter = tf.lite.Interpreter
    it = Interpreter(model_path=str(path)); it.allocate_tensors()
    i, o = it.get_input_details()[0], it.get_output_details()[0]
    ops = collections.Counter(d['op_name'] for d in it._get_ops_details())
    problems = []
    if i['dtype'] != np.int8: problems.append(f"input dtype {i['dtype'].__name__}")
    if [int(x) for x in i['shape']] != [1, IMG_SIZE, IMG_SIZE, 3]: problems.append(f"input shape {[int(x) for x in i['shape']]}")
    s, z = i['quantization']
    if not (abs(s - 1/255) < 1e-6 and z == -128): problems.append(f'input quant ({s}, {z})')
    if o['dtype'] != np.int8: problems.append(f"output dtype {o['dtype'].__name__}")
    if sorted(int(x) for x in o['shape'][1:]) != sorted([9, A]): problems.append(f"output shape {[int(x) for x in o['shape']]}")
    extra = set(ops) - FW_OPS - {'DELEGATE'}
    if extra: problems.append(f'ops not in firmware resolver: {sorted(extra)}')
    return {'problems': problems, 'input': [[int(x) for x in i['shape']], [float(s), int(z)]],
            'output': [int(x) for x in o['shape']], 'ops': dict(ops)}
def _contract():
    res = {}
    for p in sorted(EXP_DIR.rglob('*.tflite')):
        try:
            res[str(p.relative_to(EXP_DIR))] = _check_one(p)
        except Exception as e:
            res[str(p.relative_to(EXP_DIR))] = {'problems': [f'could not load: {e}']}
    for k, v in res.items():
        print(('OK   ' if not v['problems'] else 'FAIL ') + k, v['problems'] or '')
    passing = [k for k, v in res.items() if not v['problems']]
    (EXP_DIR / 'contract_check.json').write_text(json.dumps(res, indent=2))
    if passing:
        shutil.copy2(EXP_DIR / passing[0], EXP_DIR / 'FIRMWARE_READY_best_full_integer_quant.tflite')
    return {'passing': passing}
safe('firmware_contract', _contract)

In [ ]:
# Accuracy of the INT8 model itself (quantization loss vs best.pt's 0.962 mAP50).
def _val_int8():
    f = EXP_DIR / 'FIRMWARE_READY_best_full_integer_quant.tflite'
    if not f.exists():
        raise FileNotFoundError('no firmware-ready tflite to validate')
    out = {}
    for tag, path in [('int8_tflite', f), ('best_pt', EXP_DIR / 'best.pt')]:
        m = YOLO(str(path), task='detect').val(data=str(OUT / 'data.yaml'), imgsz=IMG_SIZE,
                                               split='val', batch=1, device='cpu', plots=False,
                                               project=str(EXP_DIR / 'val'), name=tag)
        out[tag] = {'mAP50': float(m.box.map50), 'mAP50-95': float(m.box.map),
                    'precision': float(m.box.mp), 'recall': float(m.box.mr),
                    'per_class_mAP50': {TARGET[i]: float(m.box.ap50[j])
                                        for j, i in enumerate(m.box.ap_class_index)}}
        print(tag, json.dumps(out[tag]))
    (EXP_DIR / 'int8_vs_pt_val.json').write_text(json.dumps(out, indent=2))
    return out
safe('val_int8_vs_pt', _val_int8)


In [ ]:
def _package():
    z = shutil.make_archive('/kaggle/working/pesolens_img256_export', 'zip', EXP_DIR)
    print('ZIP:', z, round(Path(z).stat().st_size / 1e6, 1), 'MB')
    return z
(EXP_DIR / 'REPORT.json').write_text(json.dumps(REPORT, indent=2, default=str))
safe('package', _package)
safe('cleanup_merged_dataset', lambda: shutil.rmtree(OUT) or 'removed')
print(json.dumps(REPORT, indent=2, default=str))
